# 11 — Single-cell integration: is the IAF state ONE state across cohorts?

Per-cohort analyses (notebooks 01-07) found the inflammatory fibroblast state in
Smillie (UC colon) and Martin (CD ileum) separately. This notebook tests the claim
directly: joint scVI integration of the circuit compartments (stromal + myeloid)
from both cohorts, then (1) do the two cohorts' IAF cells co-cluster in joint
space, (2) do joint-space IAF shares by cohort x condition reproduce the
per-cohort numbers (Smillie 0.24 / 7.08 / 12.90%; Martin combined 0.19 -> 44.32%),
(3) Kinchen (pre-normalized, cannot enter the counts model) is projected at
program-score level only. Fibroblast denominators follow notebook 07:
Smillie = stromal compartment; Martin = fibroblast cluster 10 descendants.
Runtime note: scVI on ~100k cells, CPU — expect roughly 30-60 minutes.


In [3]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import scanpy as sc, anndata as ad
sc.settings.verbosity = 1
DATA = Path.home() / "Documents/immune-endotypes/data"
MOD = Path.home() / "Documents/github/immune-endotypes-portfolio/modules/05_cross_cohort"
PROG = ["IL11","IL13RA2","IL24","CHI3L1","MMP3","MMP10","IL6","CXCL1","CXCL5"]

sm = sc.read_h5ad(DATA / "smillie_uc/adata_qc.h5ad")
print("Smillie:", sm.shape, "| obs:", list(sm.obs.columns))
print("counts layer:", "counts" in sm.layers, "| X max:", float(sm.X[:2000].max()))
ma = sc.read_h5ad(DATA / "martin_2019/adata_martin_qc.h5ad")
print("Martin:", ma.shape, "| obs:", list(ma.obs.columns))
print("counts layer:", "counts" in ma.layers, "| X max:", float(ma.X[:2000].max()))


Smillie: (351952, 18172) | obs: ['Origin', 'Subject', 'Sample', 'Location', 'Replicate', 'Health', 'Cluster', 'nGene', 'nUMI', 'Major_l1', 'Major_l2', 'Major_l3', 'Major_l4', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'leiden_0.5']
counts layer: True | X max: 8.800158500671387
Martin: (115029, 25356) | obs: ['condition', 'patient', 'sample', 'dataset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'leiden_0.5']
counts layer: True | X max: 8.952227592468262


In [4]:
# --- compartment selection + harmonized obs ---
# Smillie (original column names on disk): Health / Major_l1 / Major_l2 / Cluster
sm.obs["condition"] = sm.obs["Health"].astype(str)
sm.obs["cohort"] = "Smillie"
sm.obs["compartment"] = np.where(sm.obs["Major_l1"].astype(str) == "Stromal", "Stromal",
                          np.where(sm.obs["Major_l2"].astype(str) == "Myeloid", "Myeloid", "Other"))
sm.obs["denom"] = (sm.obs["Major_l1"].astype(str) == "Stromal")   # nb07 denominator
sm_c = sm[sm.obs["compartment"].isin(["Stromal", "Myeloid"])].copy()
print("Smillie circuit cells:", sm_c.shape, sm_c.obs["compartment"].value_counts().to_dict())

# Martin (nb03 labels): fibroblast = leiden 10, myeloid = leiden 4
ma.obs["cohort"] = "Martin"
ma.obs["compartment"] = np.where(ma.obs["leiden_0.5"].astype(str) == "10", "Fibroblast",
                          np.where(ma.obs["leiden_0.5"].astype(str) == "4", "Myeloid", "Other"))
ma.obs["denom"] = (ma.obs["leiden_0.5"].astype(str) == "10")
ma_c = ma[ma.obs["compartment"].isin(["Fibroblast", "Myeloid"])].copy()
print("Martin circuit cells:", ma_c.shape, ma_c.obs["compartment"].value_counts().to_dict())

keep = ["cohort", "condition", "compartment", "denom"]
for a in (sm_c, ma_c):
    a.obs = a.obs[[c for c in keep if c in a.obs.columns]].copy()
    if "counts" in a.layers: a.X = a.layers["counts"].copy()
    a.obs_names_make_unique()
joint = ad.concat([sm_c, ma_c], join="inner", label="cohort", keys=["Smillie", "Martin"])
print("joint:", joint.shape, "| genes shared:", joint.n_vars)
joint.layers["counts"] = joint.X.copy()


Smillie circuit cells: (59099, 18172) {'Stromal': 31829, 'Myeloid': 27270}
Martin circuit cells: (9377, 25356) {'Myeloid': 6435, 'Fibroblast': 2942}
joint: (68476, 16632) | genes shared: 16632


In [5]:
# --- program score on normalized copy, then scVI on counts ---
tmp = joint.copy()
sc.pp.normalize_total(tmp, target_sum=1e4); sc.pp.log1p(tmp)
sc.tl.score_genes(tmp, PROG, score_name="iaf_score")
joint.obs["iaf_score"] = tmp.obs["iaf_score"].values
print(joint.obs.groupby(["cohort", "condition"])["iaf_score"].mean())

sc.pp.highly_variable_genes(joint, flavor="seurat_v3", n_top_genes=3000,
                            batch_key="cohort", layer="counts")
hv = joint[:, joint.var["highly_variable"]].copy()
import scvi
scvi.model.SCVI.setup_anndata(hv, layer="counts", batch_key="cohort")
model = scvi.model.SCVI(hv, n_latent=30)
model.train(max_epochs=25)
joint.obsm["X_scVI"] = model.get_latent_representation(hv)
print("latent:", joint.obsm["X_scVI"].shape)


cohort   condition   
Smillie  Healthy        -0.057242
         Inflamed        0.006840
         Involved             NaN
         Non-inflamed   -0.012793
         Uninvolved           NaN
Martin   Healthy              NaN
         Inflamed             NaN
         Involved        0.004972
         Non-inflamed         NaN
         Uninvolved     -0.056196
Name: iaf_score, dtype: float64


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Epoch 25/25: 100%|████| 25/25 [08:09<00:00, 18.62s/it, v_num=1, train_loss=624]

`Trainer.fit` stopped: `max_epochs=25` reached.


Epoch 25/25: 100%|████| 25/25 [08:09<00:00, 19.58s/it, v_num=1, train_loss=624]
latent: (68476, 30)


In [6]:
sc.pp.neighbors(joint, use_rep="X_scVI", n_neighbors=15)
sc.tl.umap(joint)
sc.tl.leiden(joint, resolution=0.6, key_added="joint_leiden")
print("joint clusters:", joint.obs["joint_leiden"].nunique())

# cluster summary: size, cohort mix, fibroblast share, mean IAF score
rows = []
for cl, idx in joint.obs.groupby("joint_leiden").groups.items():
    o = joint.obs.loc[idx]
    rows.append({"cluster": cl, "n": len(o),
                 "frac_martin": round(float((o["cohort"] == "Martin").mean()), 3),
                 "frac_denom": round(float(o["denom"].mean()), 3),
                 "mean_iaf_score": round(float(o["iaf_score"].mean()), 3)})
ct = pd.DataFrame(rows).sort_values("mean_iaf_score", ascending=False)
print(ct.to_string(index=False))
fib_clusters = ct[ct["frac_denom"] > 0.5]
iaf_cluster = fib_clusters.iloc[0]["cluster"]
print("JOINT IAF cluster:", iaf_cluster)
joint.obs["is_joint_iaf"] = (joint.obs["joint_leiden"] == iaf_cluster)
ct.to_csv(MOD / "results/integration_clusters.csv", index=False)


joint clusters: 26
cluster    n  frac_martin  frac_denom  mean_iaf_score
     16 1093        0.145       1.000           0.822
     23  329        0.000       1.000           0.664
      8 3983        0.009       0.995           0.228
     21  442        0.998       1.000           0.208
     20  808        0.002       1.000           0.195
     19  927        0.000       1.000           0.074
     17  947        0.000       1.000           0.017
     10 2644        0.407       0.000           0.010
      0 6644        0.028       1.000          -0.006
     11 2334        0.439       0.998          -0.006
      1 6270        0.117       0.000          -0.035
     25   74        0.000       0.000          -0.037
     22  428        0.000       0.009          -0.045
      6 4885        0.013       0.000          -0.052
      5 5312        0.001       0.001          -0.057
      7 4464        0.036       1.000          -0.062
      9 3210        0.002       1.000          -0.065
      3 5

In [7]:
# --- agreement check: joint-space IAF share vs notebook 07 numbers ---
joint.obs["denom_fib"] = joint.obs["denom"]
tab = (joint.obs[joint.obs["denom_fib"]]
       .groupby(["cohort", "condition"])["is_joint_iaf"].mean().mul(100).round(2)
       .rename("joint_iaf_share_pct").reset_index())
print(tab.to_string(index=False))
print("per-cohort reference (nb07): Smillie Healthy 0.24 / Non-inflamed 7.08 / Inflamed 12.90 ; Martin Uninvolved 0.19 / Involved 44.32 (IAF+CHI3L1 combined)")
tab.to_csv(MOD / "results/integration_check.csv", index=False)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, key, title in zip(axes,
        ["cohort", "condition", "iaf_score", "is_joint_iaf"],
        ["cohort", "condition", "IAF program score", "joint IAF cluster"]):
    sc.pl.umap(joint, color=key, ax=ax, show=False, title=title, size=4)
plt.tight_layout()
plt.savefig(MOD / "figures/integration_umap.png", dpi=150, bbox_inches="tight")
print("saved figures/integration_umap.png")
out = DATA / "integrated"; out.mkdir(parents=True, exist_ok=True)
joint.write_h5ad(out / "adata_integrated_circuit.h5ad")
print("saved adata_integrated_circuit.h5ad")


 cohort    condition  joint_iaf_share_pct
Smillie      Healthy                 0.00
Smillie     Inflamed                 7.89
Smillie     Involved                  NaN
Smillie Non-inflamed                 0.97
Smillie   Uninvolved                  NaN
 Martin      Healthy                  NaN
 Martin     Inflamed                  NaN
 Martin     Involved                11.81
 Martin Non-inflamed                  NaN
 Martin   Uninvolved                 0.06
per-cohort reference (nb07): Smillie Healthy 0.24 / Non-inflamed 7.08 / Inflamed 12.90 ; Martin Uninvolved 0.19 / Involved 44.32 (IAF+CHI3L1 combined)
saved figures/integration_umap.png
saved adata_integrated_circuit.h5ad


In [8]:
# --- Kinchen: score-level projection only (pre-normalized data) ---
ki = sc.read_h5ad(DATA / "kinchen_2018/adata_kinchen_human_qc.h5ad")
print("Kinchen:", ki.shape, "| obs:", list(ki.obs.columns))
print(ki.obs.head(3).to_string())


Kinchen: (9271, 21171) | obs: ['condition', 'sample', 'dataset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'leiden_0.5']
                     condition sample      dataset  n_genes_by_counts  log1p_n_genes_by_counts  total_counts  log1p_total_counts leiden_0.5
AAACCTGCAAGGACTG-S66   Healthy    S66  Kinchen2018               2313                 7.746733   2775.123989            7.928811          1
AAACCTGCACAACTGT-S66   Healthy    S66  Kinchen2018               1133                 7.033506   2151.228692            7.674259          1
AAACCTGCACCCTATC-S66   Healthy    S66  Kinchen2018               1449                 7.279319   2298.312566            7.740365          1


In [9]:
import numpy as np, pandas as pd
import scanpy as sc
col = ki[:, "COL1A1"].X
colv = np.asarray(col.todense()).ravel() if hasattr(col, "todense") else np.asarray(col).ravel()
ki.obs["col1a1_pos"] = colv > 0
frac = ki.obs.groupby("leiden_0.5")["col1a1_pos"].mean()
fib_clusters = list(frac[frac > 0.5].index.astype(str))
print("fibroblast clusters (COL1A1 frac > 0.5):", fib_clusters)
kf = ki[ki.obs["leiden_0.5"].astype(str).isin(fib_clusters)].copy()
genes = [g for g in PROG if g in kf.var_names]
print("program genes present:", genes)
sc.tl.score_genes(kf, genes, score_name="iaf_score")
thr = float(joint.obs.loc[joint.obs["is_joint_iaf"], "iaf_score"].median())
print("joint IAF cluster median score (threshold):", round(thr, 3))
rows = []
for cond, idx in kf.obs.groupby("condition").groups.items():
    o = kf.obs.loc[idx]
    rows.append({"condition": cond, "n_fibroblasts": len(o),
                 "mean_iaf_score": round(float(o["iaf_score"].mean()), 3),
                 "share_above_joint_iaf_median_pct": round(float((o["iaf_score"] > thr).mean()) * 100, 2)})
kp = pd.DataFrame(rows)
print(kp.to_string(index=False))
kp.to_csv(MOD / "results/integration_kinchen_projection.csv", index=False)
print("saved results/integration_kinchen_projection.csv")


fibroblast clusters (COL1A1 frac > 0.5): ['0', '1', '2', '3', '5', '7', '8', '9', '11', '14']
program genes present: ['IL11', 'IL13RA2', 'IL24', 'CHI3L1', 'MMP3', 'MMP10', 'IL6', 'CXCL1', 'CXCL5']
joint IAF cluster median score (threshold): 0.768
condition  n_fibroblasts  mean_iaf_score  share_above_joint_iaf_median_pct
  Healthy           3654          -0.213                              0.00
       UC           3602           0.147                              8.25
saved results/integration_kinchen_projection.csv


Kinchen cell (run after inspecting the obs print above): select fibroblast
clusters by the notebook-02 rule (whole-data clusters with COL1A1 fraction > 0.5),
score the 9-gene program (data are already log-normalized), and compare the score
distribution by condition with the joint IAF cluster's score range from cell 5.
If the obs column names differ from the Smillie/Martin pattern, paste the print
output back and the cell gets patched to the real names.
